# 04c LLM pipeline 2: visualisations to a natural language explanation

Idea: the LLM receives waterfall plots of the SHAP/EBM contributions as an image
and explains what it sees, without structured JSON data.

* Advantage: uses visual pattern recognition, closer to human intuition.
* Disadvantage: precise numbers are lost in the image.

Flow:
1. Generate plots and save them in explanations/plots/ (no API key needed)
2. Send plots plus a context prompt to Claude
3. Save results in results/pipeline05/

In [ ]:
from __future__ import annotations

import sys, json, time
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import joblib
import shap

from utils import (
    INSTANCE_IDS,
    EXPLANATIONS_DIR, RESULTS_DIR, MODELS_DIR, PROMPTS_DIR,
)
from utils.data import load_train_test
from utils.llm import ask_with_images, DEFAULT_MODEL, MAX_TOKENS_GENERATION, strip_scratchpad

LOSS_KEY  = 'poisson_log'
MODEL     = DEFAULT_MODEL
MAX_TOKENS = MAX_TOKENS_GENERATION

# n=20 validity run: the 10 instances, 1 generation, real time. Plots and results
# go directly into the main folders (explanations/plots/, results/pipeline05/).
GEN_INSTANCE_IDS = INSTANCE_IDS
N_GEN            = 1
PLOTS_DIR        = EXPLANATIONS_DIR / 'plots'
OUT_DIR          = RESULTS_DIR / 'pipeline05'
PLOTS_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'LLM model:     {MODEL}')
print(f'Instances:     {len(GEN_INSTANCE_IDS)} x {N_GEN} generation')
print(f'Plots output:  {PLOTS_DIR}')
print(f'Results:       {OUT_DIR}')

## 1. Load data and models

In [ ]:
X_train, y_train, X_test, y_test = load_train_test()
from utils.models import load_models
xgb, ebm = load_models(LOSS_KEY)

# Initialise the SHAP TreeExplainer once
shap_explainer = shap.TreeExplainer(xgb)
shap_values    = shap_explainer(X_test)

print('Models and SHAP values loaded.')
print(f'X_test shape: {X_test.shape}')

## 2. Plot generation

For each instance and model a waterfall plot is created:
* XGBoost: shap.plots.waterfall (built in)
* EBM: matplotlib bar chart (interpret uses Plotly, not matplotlib)

In [ ]:
def plot_xgb_waterfall(instance_id: int) -> Path:
    """SHAP waterfall plot for XGBoost, saved as PNG."""
    pos  = X_test.index.get_loc(instance_id)
    pred = float(xgb.predict(X_test.iloc[[pos]])[0])
    y    = float(y_test.iloc[pos])

    shap.plots.waterfall(shap_values[pos], show=False, max_display=10)
    plt.title(f'XGBoost | instance {instance_id} | '
              f'prediction: {pred:.1f}  (actual: {y:.0f})', pad=12)
    path = PLOTS_DIR / f'waterfall_xgb_{LOSS_KEY}_inst{instance_id}.png'
    plt.savefig(path, dpi=130, bbox_inches='tight')
    plt.close('all')
    return path


def plot_ebm_waterfall(instance_id: int) -> Path:
    """Manual waterfall plot for the EBM contributions."""
    pos  = X_test.index.get_loc(instance_id)
    inst = X_test.iloc[[pos]]
    pred = float(ebm.predict(inst)[0])
    y    = float(y_test.iloc[pos])

    exp  = ebm.explain_local(inst)
    d    = exp.data(0)
    base = float(d['extra']['scores'][0])  # intercept

    # Main effects only (no interactions), top 10 by absolute score
    pairs = [(n, float(s)) for n, s in zip(d['names'], d['scores'])]
    pairs = sorted(pairs, key=lambda x: abs(x[1]), reverse=True)[:10]
    labels = [p[0] for p in pairs]
    scores = [p[1] for p in pairs]
    colors = ['#d73027' if v > 0 else '#4575b4' for v in scores]

    fig, ax = plt.subplots(figsize=(8, 5))
    bars = ax.barh(labels[::-1], scores[::-1], color=colors[::-1], height=0.65)
    ax.axvline(0, color='black', lw=0.8)
    for bar, val in zip(bars, scores[::-1]):
        ax.text(val + (0.01 if val >= 0 else -0.01), bar.get_y() + bar.get_height()/2,
                f'{val:+.3f}', va='center', ha='left' if val >= 0 else 'right', fontsize=8)
    ax.set_xlabel('Contribution (log scale)', fontsize=10)
    ax.set_title(f'EBM | instance {instance_id} | '
                 f'prediction: {pred:.1f}  (actual: {y:.0f})\n'
                 f'Base value (intercept): {base:.3f}', fontsize=11, pad=12)
    plt.tight_layout()
    path = PLOTS_DIR / f'waterfall_ebm_{LOSS_KEY}_inst{instance_id}.png'
    plt.savefig(path, dpi=130, bbox_inches='tight')
    plt.close('all')
    return path

In [ ]:
plot_paths: dict[tuple, Path] = {}

# Plots are deterministic per (model, instance), so 1 plot per unit regardless of
# the number of generations (10 instances x 2 models = 20 plots, no API call).
for n_done, iid in enumerate(GEN_INSTANCE_IDS, 1):
    p_xgb = plot_xgb_waterfall(iid)
    p_ebm = plot_ebm_waterfall(iid)
    plot_paths[('xgb', iid)] = p_xgb
    plot_paths[('ebm', iid)] = p_ebm
    if n_done <= 5 or n_done % 25 == 0:
        print(f'  [{n_done:3d}/{len(GEN_INSTANCE_IDS)}] inst={iid:4d}  to {p_xgb.name} / {p_ebm.name}')

print(f'\n{len(plot_paths)} plots saved in {PLOTS_DIR}')

### Methodological limitation: plot resolution as a confound

The plots are saved at dpi=130. A lower faithfulness (RA/SA/VA) of the vision
pipeline compared to JSON to Text is therefore not necessarily inherent to the
visual modality: at dpi=130 the bar labels and feature values can be hard to read
for the LLM, especially for similarly long bars. This overlays the actual modality
effect with a measurement artefact.

Documented as a limitation in results/limitations_vision_resolution.txt. Comparisons
between vision and the other pipelines should be reported with this caveat.

## 3. Show example plots

In [ ]:
from IPython.display import display, Image as IPImage

for model_name in ['xgb', 'ebm']:
    iid = GEN_INSTANCE_IDS[0]
    print(f'{model_name.upper()} instance {iid}:')
    display(IPImage(filename=str(plot_paths[(model_name, iid)]), width=620))
    print()

## 4. System prompt

In [ ]:
SYSTEM_PROMPT = (PROMPTS_DIR / "pipeline_05_vision.md").read_text()

print(f'System prompt: {len(SYSTEM_PROMPT)} characters, ~{len(SYSTEM_PROMPT)//4} tokens (estimated)')

## 5. LLM calls with plots

In [ ]:
from utils import (
    run_resumable_generation, build_generation_record, load_local_explanation,
)
from utils.llm import build_image_params, run_params
from utils.explanations import (
    WEEKDAY_NAMES, MONTH_NAMES, WEATHER_NAMES,
    TEMP_FACTOR, HUM_FACTOR, WIND_FACTOR,
)

# n=20 validity: real time over the central resume loop. build_context_prompt
# stays visible (the method: which text accompanies the plot). IO loading and the
# record schema come from utils centrally (golden test in test_generation_loop.py).

def build_context_prompt(model_name: str, instance_id: int) -> str:
    """Context text for the plot: readable feature values from the JSON explanation."""
    l = load_local_explanation(model_name, instance_id, loss_key=LOSS_KEY,
                               explanations_dir=EXPLANATIONS_DIR)
    fv = l["feature_values"]
    lines = [
        f"Model: {model_name.upper()}",
        f"Instance ID: {instance_id}",
        f"Time: {int(fv['hr']):02d}:00",
        f"Weekday: {WEEKDAY_NAMES[int(fv['weekday'])]}",
        f"Month: {MONTH_NAMES[int(fv['mnth'])]}",
        f"Year: {'2011' if int(fv['yr']) == 0 else '2012'}",
        f"Weather: {WEATHER_NAMES.get(int(fv['weathersit']), str(fv['weathersit']))}",
        f"Temperature: ~{float(fv['temp']) * TEMP_FACTOR:.1f} C (normalised: {float(fv['temp']):.2f})",
        f"Humidity: {float(fv['hum']) * HUM_FACTOR:.0f} %",
        f"Wind speed: {float(fv['windspeed']) * WIND_FACTOR:.1f} km/h",
        f"Holiday: {'yes' if int(fv['holiday']) == 1 else 'no'}",
        f"Actual rentals: {int(l['y_true'])} bikes",
        f"Model prediction: {l['prediction']:.1f} bikes",
        "",
        "Please explain the waterfall plot above for this situation.",
    ]
    return "\n".join(lines)


def generate_vision(model_name, iid, gen_idx):
    l_exp     = load_local_explanation(model_name, iid, loss_key=LOSS_KEY,
                                       explanations_dir=EXPLANATIONS_DIR)
    plot_path = plot_paths[(model_name, iid)]
    params = build_image_params(
        build_context_prompt(model_name, iid),
        image_paths=[plot_path],
        system=SYSTEM_PROMPT, model=MODEL, max_tokens=MAX_TOKENS, cache_system=True,
    )
    t0 = time.time()
    response = run_params(params)
    elapsed  = time.time() - t0

    text  = strip_scratchpad(response["content"][0]["text"])
    usage = response.get("usage", {})
    record = build_generation_record(
        pipeline="05_vision", model_name=model_name, instance_id=iid,
        explanation=text, usage=usage, llm_model=MODEL, loss_key=LOSS_KEY,
        prediction=l_exp["prediction"], y_true=l_exp["y_true"],
        elapsed_s=round(elapsed, 2), extra={"plot_file": plot_path.name},
    )
    u = record["usage"]
    print(f"  {model_name.upper()} inst={iid:4d} g{gen_idx}  "
          f"pred={record['prediction']:6.1f}  y={record['y_true']:5.0f}  "
          f"in={u['input_tokens']}  out={u['output_tokens']}  "
          f"cache={u['cache_read_input_tokens']}  t={elapsed:.1f}s")
    return record


results = run_resumable_generation(
    model_names=["xgb", "ebm"],
    instance_ids=GEN_INSTANCE_IDS,
    out_dir=OUT_DIR,
    generate=generate_vision,
    n_generations=N_GEN,
)

totals = {
    "in":    sum(r["usage"]["input_tokens"] for r in results),
    "out":   sum(r["usage"]["output_tokens"] for r in results),
    "cache": sum(r["usage"]["cache_read_input_tokens"] for r in results),
}
print(f"\nTotal:  input={totals['in']}  output={totals['out']}  "
      f"cache_read={totals['cache']}  ({len(results)} units)")

## 6. Example explanations

In [ ]:
for rec in results[:2]:
    sep = '=' * 70
    print(sep)
    print(f"Model: {rec['xai_model'].upper()}  |  instance: {rec['instance_id']}  "
          f"|  plot: {rec['plot_file']}")
    print(f"Prediction: {rec['prediction']:.1f}  |  actual: {rec['y_true']:.0f}")
    print(sep)
    print(rec['explanation'])
    print()

## 7. Summary

In [ ]:
import pandas as pd

summary = pd.DataFrame([
    {
        'Model':      r['xai_model'].upper(),
        'Instance':   r['instance_id'],
        'y_true':     r['y_true'],
        'Prediction': r['prediction'],
        'Words':      len(r['explanation'].split()),
        'tok_input':  r['usage']['input_tokens'],
        'tok_output': r['usage']['output_tokens'],
        'Time (s)':   r['elapsed_s'],
    }
    for r in results
])
display(summary)